# Schéma de l'entrepôt de données - Star Schema

Ce notebook fait suite à 02_preparation_uci_power.ipynb.

L'objectif est de concevoir le modèle dimensionnel (star schema) de notre entrepôt
de données, à partir des tables agrégées produites dans le notebook précédent.

Ce schéma sera ensuite implémenté dans Azure SQL Database, et servira de source
directe pour les dashboards Power BI.

## Architecture cible
data/processed/ > Lecture des tables agrégées (Parquet)

v

DimDate + DimMeter > Tables de dimensions

v

FactConsumption > Table de faits (star schema)

v

Azure SQL Database > Entrepôt de données

## Modèle dimensionnel (star schema)

Le schéma en étoile place la table de faits au centre, reliée aux tables de dimensions
qui l'entourent. C'est le modèle recommandé pour Power BI car il optimise les performances
de requête et la simplicité de navigation.

Dans notre cas :
- FactConsumption : une ligne par jour et par compteur, avec les mesures de consommation.
- DimDate : attributs temporels enrichis (année, mois, trimestre, jour de semaine, etc.).
- DimMeter : description des 3 sous-compteurs (cuisine, buanderie, chauffe-eau/clim).

## Imports et chargement

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path().resolve().parent
processed_path = project_root / "data" / "processed"

# Chargement de la table journalière (base de notre FactConsumption)
daily_df = pd.read_parquet(processed_path / "daily_power_consumption.parquet")

print("Shape :", daily_df.shape)
daily_df.head()

Shape : (1442, 8)


,Global_active_power_mean,Global_active_power_sum,Global_reactive_power_mean,Voltage_mean,Global_intensity_mean,Sub_metering_1_sum,Sub_metering_2_sum,Sub_metering_3_sum
datetime,,,,,,,,
2006-12-16,3.053475,1209.176,0.088187,236.243763,13.082828,0.0,546.0,4926.0
2006-12-17,2.354486,3390.460,0.156949,240.087028,9.999028,2033.0,4187.0,13341.0
2006-12-18,1.530435,2203.826,0.112356,241.231694,6.421667,1063.0,2621.0,14018.0
2006-12-19,1.157079,1666.194,0.104821,241.999313,4.926389,839.0,7602.0,6197.0
2006-12-20,1.545658,2225.748,0.111804,242.308062,6.467361,0.0,2648.0,14063.0


Nous rechargeons la table journalière depuis data/processed/ au format Parquet.
Le résultat confirme 1 442 lignes et 8 colonnes, identique à ce que nous avions
exporté dans le notebook 02, ce qui valide l'intégrité des fichiers produits.

Ce chargement simule la lecture que ferait un notebook Databricks depuis la zone curated
du Data Lake Azure, avant d'écrire les tables finales dans Azure SQL Database.

## Création de DimDate

La table de dimension DimDate enrichit chaque date avec des attributs temporels
(année, trimestre, mois, semaine, jour de semaine, etc.) qui permettront de filtrer et
de regrouper facilement les données dans Power BI.

C'est l'une des dimensions les plus importantes d'un star schema : elle est reliée à la
table de faits via une clé date_id (entier au format YYYYMMDD).

In [2]:
# Création de DimDate à partir des dates de daily_df
dim_date = pd.DataFrame({"date": daily_df.index})

dim_date["date_id"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.strftime("%B")
dim_date["week"] = dim_date["date"].dt.isocalendar().week.astype(int)
# 0=Lundi, 6=Dimanche
dim_date["day_of_week"] = dim_date["date"].dt.dayofweek
dim_date["day_name"] = dim_date["date"].dt.strftime("%A")
dim_date["is_weekend"] = dim_date["day_of_week"].isin([5, 6]).astype(int)

# Réorganisation avec date_id en première colonne
dim_date = dim_date[["date_id", "date", "year", "quarter", "month", "month_name", "week", "day_of_week", "day_name",
                     "is_weekend"]]

print("Shape :", dim_date.shape)
dim_date.head()

Shape : (1442, 10)


,date_id,date,year,quarter,month,month_name,week,day_of_week,day_name,is_weekend
0,20061216,2006-12-16,2006,4,12,December,50,5,Saturday,1
1,20061217,2006-12-17,2006,4,12,December,50,6,Sunday,1
2,20061218,2006-12-18,2006,4,12,December,51,0,Monday,0
3,20061219,2006-12-19,2006,4,12,December,51,1,Tuesday,0
4,20061220,2006-12-20,2006,4,12,December,51,2,Wednesday,0


La table DimDate contient 1 442 lignes (une par jour du dataset) et 10 colonnes :
une clé primaire date_id, la date brute, et 8 attributs temporels enrichis.

Lecture des premières lignes :
- Le dataset commence un samedi 16 décembre 2006 (is_weekend = 1), suivi d'un dimanche,
puis reprend en semaine à partir du 18 (lundi).
- La clé date_id (format YYYYMMDD, ex. 20061216) est un entier simple, compact et
trié naturellement, idéal comme clé de jointure entre DimDate et FactConsumption.

Rôle de cette table dans le star schema :

DimDate est la dimension la plus utilisée dans les dashboards Power BI car elle permet de
filtrer et regrouper les mesures par année, trimestre, mois, semaine ou type de jour
(semaine vs week-end).
L'attribut is_weekend sera par exemple très utile pour comparer les profils de consommation
entre jours de semaine et week-ends dans nos futurs visuels.

## Création de DimMeter

La table DimMeter décrit les 3 sous-compteurs du foyer.
C'est une petite table de référence (3 lignes) mais essentielle pour donner du sens
aux colonnes Sub_metering_1/2/3 dans la table de faits.

In [3]:
dim_meter = pd.DataFrame({
    "meter_id": [1, 2, 3],
    "meter_name": ["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"],
    "location": ["Cuisine", "Buanderie", "Chauffe-eau / Climatisation"],
    "equipment": [
        "Lave-vaisselle, four, micro-ondes",
        "Lave-linge, sèche-linge, réfrigérateur, éclairage",
        "Chauffe-eau électrique, climatisation"
    ],
    "unit": ["Wh", "Wh", "Wh"]
})

print("Shape :", dim_meter.shape)
dim_meter

Shape : (3, 5)


,meter_id,meter_name,location,equipment,unit
0,1,Sub_metering_1,Cuisine,"Lave-vaisselle, four, micro-ondes",Wh
1,2,Sub_metering_2,Buanderie,"Lave-linge, sèche-linge, réfrigérateur, éclairage",Wh
2,3,Sub_metering_3,Chauffe-eau / Climatisation,"Chauffe-eau électrique, climatisation",Wh


La table DimMeter contient 3 lignes (une par sous-compteur) et 5 colonnes :
une clé meter_id, le nom technique de la colonne source, la localisation dans le foyer,
la liste des équipements mesurés, et l'unité de mesure (Wh).

Cette table est une dimension de référence statique : elle ne changera pas dans le temps
et sert uniquement à décrire les 3 zones de mesure du foyer.

Dans Power BI, elle permettra de filtrer les visuels par zone (ex. afficher uniquement la
consommation de la cuisine ou du chauffe-eau), et d'afficher des libellés lisibles
("Cuisine", "Buanderie") plutôt que des noms techniques (Sub_metering_1).

## Création de FactConsumption

La table de faits est le coeur du star schema : chaque ligne représente un fait mesurable
(ici, la consommation d'un sous-compteur pour un jour donné) et contient les clés étrangères
vers les dimensions DimDate et DimMeter.

Notre table FactConsumption aura la granularité suivante :
1 ligne = 1 jour × 1 sous-compteur (soit 1 442 jours × 3 compteurs = 4 326 lignes).

In [4]:
# Reshape : passer de 3 colonnes Sub_metering à un format long (1 ligne par jour × compteur)
fact_df = daily_df[["Sub_metering_1_sum", "Sub_metering_2_sum", "Sub_metering_3_sum"]].copy()
fact_df.index.name = "date"
fact_df = fact_df.reset_index()

fact_long = fact_df.melt(
    id_vars="date",
    value_vars=["Sub_metering_1_sum", "Sub_metering_2_sum", "Sub_metering_3_sum"],
    var_name="meter_name",
    value_name="energy_wh"
)

# Nettoyage du nom de compteur (retirer le suffixe _sum)
fact_long["meter_name"] = fact_long["meter_name"].str.replace("_sum", "", regex=False)

# Ajout des clés étrangères
fact_long["date_id"] = fact_long["date"].dt.strftime("%Y%m%d").astype(int)
fact_long = fact_long.merge(dim_meter[["meter_id", "meter_name"]], on="meter_name", how="left")

# Ajout des mesures globales journalières
fact_long = fact_long.merge(
    daily_df[["Global_active_power_mean", "Global_active_power_sum", "Global_reactive_power_mean", "Voltage_mean",
              "Global_intensity_mean"]].reset_index().rename(columns={"datetime": "date"}),
    on="date",
    how="left"
)

# Réorganisation des colonnes
fact_consumption = fact_long[[
    "date_id", "meter_id", "date",
    "energy_wh",
    "Global_active_power_mean", "Global_active_power_sum",
    "Global_reactive_power_mean", "Voltage_mean", "Global_intensity_mean"
]].sort_values(["date_id", "meter_id"]).reset_index(drop=True)

print("Shape :", fact_consumption.shape)
fact_consumption.head()

Shape : (4326, 9)


,date_id,meter_id,date,energy_wh,Global_active_power_mean,Global_active_power_sum,Global_reactive_power_mean,Voltage_mean,Global_intensity_mean
0,20061216,1,2006-12-16,0.0,3.053475,1209.176,0.088187,236.243763,13.082828
1,20061216,2,2006-12-16,546.0,3.053475,1209.176,0.088187,236.243763,13.082828
2,20061216,3,2006-12-16,4926.0,3.053475,1209.176,0.088187,236.243763,13.082828
3,20061217,1,2006-12-17,2033.0,2.354486,3390.460,0.156949,240.087028,9.999028
4,20061217,2,2006-12-17,4187.0,2.354486,3390.460,0.156949,240.087028,9.999028


La table de faits contient 4 326 lignes (1 442 jours × 3 compteurs) et 9 colonnes,
structurées autour de deux clés étrangères et de mesures de consommation.

Structure de la table :

- date_id > clé étrangère vers DimDate
- meter_id > clé étrangère vers DimMeter
- energy_wh > énergie consommée par le sous-compteur ce jour-là (en Wh)
- Les 5 colonnes restantes (Global_active_power_mean/sum, Global_reactive_power_mean, Voltage_mean,
Global_intensity_mean) > mesures globales du foyer pour ce jour, répétées pour chaque compteur (dénormalisation
typique d'un star schema).

Lecture des premières lignes :

- Le 16 décembre 2006 montre que Sub_metering_1 (cuisine) = 0 Wh : aucun appareil
de cuisine n'a été utilisé ce soir-là, alors que Sub_metering_3 (chauffe-eau/clim)
= 4 926 Wh, confirmant un fort usage du chauffage en ce début d'hiver.
- Le 17 décembre 2006 (premier jour complet) montre les 3 sous-compteurs actifs :
cuisine (2 033 Wh), buanderie (4 187 Wh) et chauffe-eau (13 341 Wh), avec une puissance
active globale de ≈ 2,35 kW en moyenne sur la journée.

Cette granularité 1 ligne = 1 jour × 1 compteur est idéale pour Power BI :
elle permet de filtrer par zone (DimMeter), par période (DimDate), et d'agréger
facilement la consommation totale ou par sous-compteur sur n'importe quelle période.n

## Export des tables du schéma

Nous exportons les 3 tables du star schema (DimDate, DimMeter, FactConsumption)
en CSV dans un dossier data/schema/, qui représente la couche finale de notre entrepôt
de données avant le chargement dans Azure SQL Database.

In [5]:
schema_path = project_root / "data" / "schema"
schema_path.mkdir(parents=True, exist_ok=True)

dim_date.to_csv(schema_path / "DimDate.csv", index=False)
dim_meter.to_csv(schema_path / "DimMeter.csv", index=False)
fact_consumption.to_csv(schema_path / "FactConsumption.csv", index=False)

print("Export terminé. Fichiers générés :")
for f in sorted(schema_path.iterdir()):
    size_kb = round(f.stat().st_size / 1024, 1)
    print(f"  {f.name} - {size_kb} KB")

Export terminé. Fichiers générés :
  DimDate.csv - 73.8 KB
  DimMeter.csv - 0.3 KB
  FactConsumption.csv - 468.9 KB


## Conclusion

Ce notebook a concrétisé le modèle dimensionnel (star schema) de notre plateforme BI
de suivi de la consommation énergétique.

Ce que nous avons accompli :

1. Rechargement des tables agrégées depuis data/processed/ (Parquet).
2. DimDate : 1 442 lignes, 10 attributs temporels (année, trimestre, mois, semaine, jour de semaine, is_weekend),
avec une clé date_id au format YYYYMMDD.
3. DimMeter : 3 lignes décrivant les sous-compteurs (cuisine, buanderie, chauffe-eau/climatisation), avec
localisation et équipements associés.
4. FactConsumption : 4 326 lignes à la granularité 1 jour × 1 compteur, portant
l'énergie consommée par zone et les mesures globales du foyer.
5. Export des 3 tables en CSV dans data/schema/, prêtes pour Azure SQL Database.

Schéma final :

FactConsumption

- date_id > DimDate.date_id

- meter_id > DimMeter.meter_id

- energy_wh

- Global_active_power_mean

- Global_active_power_sum

- Global_reactive_power_mean

- Voltage_mean

- Global_intensity_mean

Le notebook 04_chargement_azure_sql.ipynb abordera le chargement de ces 3 tables
dans Azure SQL Database via une connexion Python (pyodbc / SQLAlchemy),
ce qui complètera la couche "entrepôt" de notre architecture Azure.